# 📱 Практикум: рынок телефонов Lalafo

### Линейная алгебра и статистика на живых — и очень грязных — данных

Перед вами **8 160 реальных объявлений** о продаже телефонов с доски Lalafo (Кыргызстан).
Данные настоящие, а значит — грязные: перепутанные валюты, цены в два миллиарда сом,
«здоровье батареи 123 миллиона процентов», перекупы, спам и смесь трёх языков в заголовках.

Это не баг, это **самое интересное**. Именно на грязных данных видно разницу между тем,
- как на цифры смотрит **обыватель** («айфоны дороже, чего тут думать»), и
- что видит **исследователь** («секунду, а половина этих цен — вообще не цены»).

Ваша задача в каждом кейсе — не «посчитать среднее», а **принять решение и защитить его графиками**.

## 🎯 Правила игры

1. **Пять кейсов** (T1…T5). Каждый — это житейское решение, где на кону деньги.
2. В начале кейса **запишите свой наивный ответ** (одна строка, честно, до анализа). Потом проверите, обманула ли вас интуиция.
3. Стройте **много графиков**. График, который меняет ваше мнение, — это удача, а не поражение.
4. Финал каждого кейса — **вердикт**: одно решение + 3–5 графиков, которые его доказывают.
5. В конце — **защита**: 3 минуты у доски. Не пересказ формул, а «я рекомендую вот это, и вот почему».

**Две планки в каждом кейсе:**
- 🟢 *Минимум* — довести до вердикта хотя бы по основному признаку.
- 🔵 *Если зашло* — копнуть глубже (доп. признаки, спорные случаи, красивая визуализация).

> Загрузка данных и черновая обработка признаков **уже сделаны за вас** (ячейки ниже).
> Не тратьте время на парсинг строк — тратьте его на размышление.

---
## ⚙️ Шаг 0. Настройка окружения

Просто запустите эту ячейку. Ставить ничего не нужно — всё есть в Colab.

In [ ]:
import numpy as np, pandas as pd, re
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (8, 5), "figure.dpi": 110,
                     "axes.titlesize": 13, "axes.grid": True})
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
print("Окружение готово.")

## 📥 Шаг 1. Загрузка датасета

Берём **только таблицы объявлений и продавцов** — фотографии (≈5 ГБ) не качаем, они здесь не нужны.

In [ ]:
from datasets import load_dataset

REPO = "aiacademy-kg/lalafo-kg-phones"
# config "listings" = ads, "users" = sellers. We never touch config "images".
raw   = load_dataset(REPO, "listings", split="train").to_pandas()
users = load_dataset(REPO, "users",    split="train").to_pandas()
print("Объявлений:", len(raw), "| продавцов:", len(users))

## 🧹 Шаг 2. Черновой инжиниринг признаков (готово за вас)

Здесь мы чиним **типы данных** и убираем **очевидный** мусор (например, батарею на 123 млн %).
А вот решение, что считать выбросом в **ценах**, мы намеренно оставляем вам — это и есть часть кейсов.

Прочитайте комментарии: полезно понимать, откуда взялась каждая новая колонка.

In [ ]:
# --- Rough feature engineering: fix TYPES and obvious garbage only. ----------
# We deliberately DO NOT drop price outliers here: deciding "garbage vs real"
# is YOUR job in the cases below.

USD_KGS = 89.0   # approximate 2024-2025 rate; good enough for rough comparison

def _first_num(x):
    """Pull the first number out of a messy string like '256 ГБ' or '< 2 ГБ'."""
    if not isinstance(x, str):
        return np.nan
    m = re.search(r"(\d+(?:[.,]\d+)?)", x)
    return float(m.group(1).replace(",", ".")) if m else np.nan

def _storage_gb(x):
    v = _first_num(x)
    if isinstance(x, str) and ("тб" in x.lower() or "tb" in x.lower()):
        v *= 1024                        # 1 ТБ -> 1024 ГБ
    return v

def _battery_pct(x):
    v = _first_num(x)
    return v if (v == v and 0 <= v <= 100) else np.nan   # kill impossible values

_EMOJI = re.compile(r"[\U0001F000-\U0001FAFF☀-➿]")

def engineer(listings, users):
    df = listings.copy()

    # 1) money in ONE currency (KGS). Unknown currency -> NaN (can't compare).
    price = df["price"].astype(float)
    df["price_kgs"] = np.select(
        [df["currency"].eq("KGS"), df["currency"].eq("USD")],
        [price, price * USD_KGS],
        default=np.nan,
    )

    # 2) specs: strings -> numbers
    df["storage_gb"]  = df["storage"].map(_storage_gb)
    df["ram_gb"]      = df["ram"].map(_first_num)
    df["battery_pct"] = df["battery_health_pct"].map(_battery_pct)

    # 3) brand / model helpers
    df["is_iphone"]  = df["brand"].eq("Apple iPhone")
    df["iphone_gen"] = df["model"].str.extract(r"iPhone\s*(\d+)")[0].astype(float)
    df.loc[~df["is_iphone"], "iphone_gen"] = np.nan
    df["condition_norm"] = df["condition"]           # already {'Новый','Б/у',NaN}

    # 4) engagement ratios (guard against divide-by-zero)
    v = df["views"].replace(0, np.nan)
    df["conv_rate"] = df["callers_count"] / v        # calls per view
    df["fav_rate"]  = df["favorite_count"] / v       # favorites per view

    # 5) seller signals: how many ads from this user IN THIS dataset ...
    df["n_ads_in_data"] = df.groupby("user_id")["ad_id"].transform("size")
    # ... plus their profile from the users table
    u = users.drop_duplicates("user_id").set_index("user_id")
    df["user_ads_count"]       = df["user_id"].map(u["ads_count"])
    df["seller_response_rate"] = df["user_id"].map(u["response_rate"])
    df["seller_pro"]           = df["user_id"].map(u["pro"])

    # 6) cheap text signals
    t = df["title"].fillna("")
    df["title_len"]       = t.str.len()
    # use Python re (not the arrow/RE2 backend, which rejects \U-escapes)
    df["title_has_emoji"] = t.map(lambda s: bool(_EMOJI.search(s)))
    return df

df = engineer(raw, users)
print("Готово:", df.shape[0], "объявлений,", df.shape[1], "колонок")
df[["title", "price_kgs", "brand", "storage_gb", "ram_gb",
    "battery_pct", "conv_rate", "n_ads_in_data"]].head()

## 📖 Шаг 3. Обзор датасета: что за признаки и что они значат

Одна строка = одно объявление. Ниже — самые полезные колонки. ⭐ помечены те, что мы **добавили сами** на шаге 2.

### Про телефон
| Колонка | Что это | Подвох |
|---|---|---|
| `title`, `description` | заголовок и текст объявления | смесь русского/кыргызского/латиницы, эмодзи, опечатки |
| `brand`, `model` | бренд и модель | `Apple iPhone` — почти половина доски |
| `condition` | `Новый` / `Б/у` | пусто у ~36% |
| `price`, `currency` | цена и валюта как в объявлении | валюты **смешаны** (KGS/USD), у 12% цены нет |
| ⭐ `price_kgs` | цена, приведённая к сомам | выбросы **не убраны** — это ваша работа |
| `storage`, `ram`, `battery_health_pct` | «256 ГБ», «8 ГБ», «84» — **строки** | «< 2 ГБ», «1 ТБ», мусорные значения |
| ⭐ `storage_gb`, `ram_gb`, `battery_pct` | те же, но **числами** | battery почищена до диапазона 0–100 |
| ⭐ `is_iphone`, `iphone_gen` | флаг айфона и его поколение (13, 14, …) | только для iPhone |

### Внимание / продавец / гео
| Колонка | Что это | Подвох |
|---|---|---|
| `views`, `impressions` | просмотры и показы | сильный разброс |
| `favorite_count`, `callers_count`, `writers_count` | в избранное / позвонили / написали | у многих нули |
| ⭐ `conv_rate`, `fav_rate` | звонки-на-просмотр, избранное-на-просмотр | «конверсия» объявления |
| `user_id`, `is_vip`, `is_premium` | продавец и платное продвижение | у одного продавца бывает 100+ объявлений |
| ⭐ `n_ads_in_data`, `user_ads_count` | сколько объявлений у продавца (в выборке / всего) | большое число = магазин/перекуп |
| ⭐ `seller_response_rate`, `seller_pro` | как быстро отвечает, PRO-аккаунт | из таблицы `users` |
| `city`, `region`, `lat`, `lng` | город и координаты | Бишкек — больше половины |

> 🔎 **Легенда для затравки.** В сыром `battery_health_pct` попался ввод `123 658 674`. Из-за
> одного этого числа *среднее* «здоровье батареи» по доске равнялось **49 333 %** вместо
> честных **86 %**. Один человек с опечаткой — и статистика сломана. Мы это значение убрали.
> А вот в ценах подобные «сюрпризы» мы оставили. Найдёте? 😉

## 🏋️ Разминка (не оценивается)

Чтобы размять руки перед кейсами — вот **готовый шаблон** «сгруппировать → посчитать → нарисовать».
Скопируйте и переделывайте под свои задачи.

In [ ]:
# WARM-UP TEMPLATE: how many ads per brand? (groupby -> count -> barplot)
top = df["brand"].value_counts().head(10)

plt.figure(figsize=(8, 4.5))
sns.barplot(x=top.values, y=top.index, color="#4c72b0")
plt.xlabel("Число объявлений"); plt.ylabel("")
plt.title("Топ-10 брендов на доске")
plt.tight_layout(); plt.show()

---
# 🧩 T1. Телефоны для колл-центра

> Стартап открывает отдел обзвона. Вам выдали бюджет на 20 телефонов для операторов. Нужно не переплатить, но чтобы аппараты работали и держали заряд целую смену. Какую модель закупить?

**🙂 Наивный взгляд.** Отсортировать по цене и взять 20 самых дешёвых. Что тут думать — дешевле значит выгоднее.

**🔬 А что видит исследователь.** Самые дешёвые объявления — это ловушка. Среди них цены по «1 сом», телефоны с трещиной на экране и спам скупок. К тому же «дёшево» ≠ «выгодно»: для парка операторских трубок важнее **предсказуемость** — узкий стабильный ценник и понятное состояние, а не рекордно низкая цифра у одного объявления. (Сюрприз: здоровье батареи у бюджетных моделей почти никто не указывает — отсутствие данных тоже вывод.)

### Ваша задача
- Отберите модели-кандидаты (достаточно объявлений, чтобы вывод был не случайным).
- Решите, что считать **мусорной ценой**, и отсеките её (обоснуйте порог!).
- Сравните кандидатов по типичной цене, её **разбросу** и состоянию (`condition`).
- Вынесите вердикт: какую модель закупить и почему именно её.

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Медиана вместо среднего.** Среднее по ценам раздуто выбросами и мусором — оно врёт. Медиана устойчива: это честная «типичная цена».
- **Боксплот + правило 1.5·IQR.** Нужно, чтобы *увидеть* разброс и формально отделить выбросы от нормальных цен, а не отсекать «на глаз».
- **Перцентили.** «Дёшево, но не подозрительно» удобно ловить нижними перцентилями (например, 10–25%), а не абсолютным минимумом.
- **Ширина бокса (IQR) и доля `condition`.** Узкий бокс = предсказуемая цена (меньше риск нарваться); доля «Новый»/«Б/у» — критерий надёжности там, где батарея не указана.

### 📊 Что построить
- Боксплоты цены (`price_kgs`) по топ-моделям — сразу видно уровень, разброс и выбросы.
- Доля `condition` (Новый / Б/у) у кандидатов — столбиками.
- Итоговое сравнение финалистов: медиана цены и ширина IQR рядом.

### 🎤 Как защищать
Назовите модель и цифры: медианная цена, ширина разброса, доля Б/у. Покажите, как выглядела бы ошибка, если бы вы поверили среднему или минимальной цене.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T1: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) valid = df[df["price_kgs"].notna()]  # решите сами, что ещё отсечь как мусор
#   2) выберите модели с достаточным числом объявлений (value_counts)
#   3) sns.boxplot(...) цены по этим моделям
#   4) сравните медиану цены, ширину IQR и долю condition у кандидатов
#      (battery_pct у бюджетных моделей почти всегда пуст — это тоже вывод)
#   5) сформулируйте вердикт

---
# 🧩 T2. Гигабайты или логотип?

> Друг убеждает: «Не переплачивай за яблоко. Бери Android на 512 ГБ — это выгоднее, чем iPhone на 128». Звучит логично. Но прав ли он?

**🙂 Наивный взгляд.** Чем больше памяти — тем дороже телефон. Значит, платим мы именно за гигабайты, и друг прав: больше ГБ за те же деньги = выгоднее.

**🔬 А что видит исследователь.** Да, в среднем цена и объём памяти связаны. Но если разложить по брендам, картина переворачивается: iPhone на 128 ГБ спокойно дороже Redmi на 512 ГБ. Значит, цену двигает **бренд**, а память — уже внутри бренда. Это классическая ловушка: общая зависимость и зависимость внутри групп — **разные**.

### Ваша задача
- Проверьте наивную связь «память → цена» в целом (одним числом и графиком).
- Затем разложите по брендам и посмотрите, сохраняется ли она.
- Ответьте на вопрос друга количественно: за что мы реально платим?

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Медианная цена по объёму памяти (`groupby` + медиана).** Обыватель увидит, что с ростом памяти медиана цены ползёт вверх, и решит, что дело в гигабайтах — с этого и начните, чтобы затем оспорить. ⚠️ Сначала уберите мусорные цены (как в T1): иначе scatter превратится в кашу, а центроиды (это же средние!) уедут за выбросами.
- **Группировка по бренду (`groupby`) + медианы.** Необходима, чтобы поймать эффект «внутри бренда vs между брендами».
- **Двумерная KDE + центроиды.** Облака брендов на осях (память, цена) наглядно разъезжаются по цене — это и есть «наценка за логотип». Центроид = (средняя память, средняя цена) группы.
- **z-стандартизация.** Признаки в разных масштабах (сомы vs гигабайты) честно сравнивать нельзя. Приведите к z-шкале — и сможете сказать, на сколько «сигм» iPhone дороже при той же памяти.

### 📊 Что построить
- Scatter `storage_gb`–`price_kgs`, раскрашенный по бренду.
- Двумерная KDE (или scatter) с **центроидами** 2–3 брендов.
- Медиана цены по объёму памяти **отдельно** внутри iPhone и внутри Redmi.

### 🎤 Как защищать
Дайте другу чёткий ответ: где он прав (внутри бренда), а где нет (между брендами). Одним графиком покажите «наценку за бренд» при одинаковой памяти.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T2: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) sub = df[df["price_kgs"].notna() & df["storage_gb"].notna()]
#      совет: сначала отсеките мусорные цены (как в T1) — иначе центроиды врут
#   2) наивно: медиана цены по объёму памяти — растёт ли она с ГБ?
#      sub.groupby("storage_gb")["price_kgs"].median()
#   3) sns.scatterplot(..., hue="brand")  # раскрасьте по бренду
#   4) центроиды: sub.groupby("brand")[["storage_gb","price_kgs"]].mean()
#   5) сравните медиану цены по storage ВНУТРИ iPhone и ВНУТРИ Redmi

---
# 🧩 T3. 1000 просмотров и ноль звонков

> Вы продаёте свой телефон и хотите, чтобы объявление «работало». Но что значит «работает» — чтобы его много **смотрели** или чтобы по нему много **звонили**? Это не одно и то же.

**🙂 Наивный взгляд.** Чем больше просмотров — тем лучше объявление. Цель — набрать побольше просмотров.

**🔬 А что видит исследователь.** Просмотры и звонки связаны, но не жёстко. Есть «магниты просмотров», по которым почти не звонят: заниженная цена-приманка, платная подсветка (VIP/premium), кликбейт в заголовке. Важны не просмотры сами по себе, а **конверсия** — сколько просмотров превращается в звонки.

### Ваша задача
- Посмотрите, как связаны просмотры и звонки, и где связь ломается.
- Введите меру «эффективности» объявления и найдите, кто конвертит хорошо, а кто — только собирает просмотры.
- Проверьте: платная подсветка (`is_vip`/`is_premium`) реально помогает продать или лишь накручивает просмотры?

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Признак-отношение `conv_rate` = звонки / просмотры.** Иногда самый честный признак — не сырое число, а отношение двух.
- **Двумерное облако (views, callers) + центроиды.** Нужно, чтобы увидеть «магниты просмотров» — точки, ушедшие вправо-вниз.
- **Сравнение групп боксплотом** (VIP/premium vs обычные). Ключевой ход: сравните **и просмотры, и конверсию** — платная подсветка может поднимать просмотры, но не звонки. Тогда «платить ли за VIP» получает честный ответ.
- **Медиана vs среднее конверсии.** Снова выбросы: пара вирусных объявлений перекосит среднее.
- 🔵 *Мостик к T4:* вектор (просмотры, звонки) и его **направление** — одинаковый угол = одинаковая конверсия, независимо от масштаба. Отсюда один шаг до косинуса.

### 📊 Что построить
- Scatter `views`–`callers_count` с линией «типичной конверсии».
- Боксплот `conv_rate` для VIP/premium против обычных.
- Топ и антитоп объявлений по конверсии — что у них в заголовке?

### 🎤 Как защищать
Сформулируйте рецепт продающего объявления: на что смотреть — на просмотры или на конверсию, и стоит ли платить за VIP. Подкрепите цифрой.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T3: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) sub = df[df["views"] > 0]
#   2) sns.scatterplot(sub, x="views", y="callers_count", alpha=0.3)
#   3) сравните И просмотры, И конверсию:
#      sub.groupby("is_vip")[["views","conv_rate"]].median()
#   4) sns.boxplot(sub, x="is_vip", y="conv_rate")  # осторожно с выбросами/шкалой
#   5) посмотрите заголовки лидеров и аутсайдеров по conv_rate

---
# 🧩 T4. «Как у друга, но дешевле»

> У друга iPhone 13 Pro, и вы хотите «такой же, но дешевле». Не обязательно ту же модель — лишь бы ощущался похожим по характеристикам. Что посоветовать?

**🙂 Наивный взгляд.** Искать ровно ту же модель — iPhone 13 Pro — и ждать, пока подешевеет.

**🔬 А что видит исследователь.** «Похожий» телефон — это не обязательно та же надпись на коробке. Если описать телефон **вектором характеристик** (память, ОЗУ, батарея, цена…), то ближайшие соседи по этому вектору могут оказаться из другого бренда — и дешевле. Причём «похожесть» зависит от того, **как мерить**: евклидово расстояние и косинус дадут разные списки соседей.

### Ваша задача
- Составьте вектор признаков телефона и найдите k ближайших соседей к заданной модели.
- Сравните два определения «похожести»: евклидово расстояние и косинусная близость.
- Предложите 3 реальные альтернативы «как iPhone 13 Pro, но дешевле».

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Вектор признаков + норма.** Телефон как точка в пространстве характеристик; норма — «суммарная прокачанность».
- **z-стандартизация — обязательна, а не для красоты.** Без неё цена (десятки тысяч) раздавит память (сотни) и ОЗУ (единицы), и «похожесть» будет определяться только ценой.
- **Евклидово расстояние** — «близко по всем признакам сразу».
- **Скалярное произведение → косинус** — «похоже по *профилю*/направлению», масштаб не важен. Именно косинус ловит «тот же баланс характеристик». Поэкспериментируйте: включать ли цену в вектор?
- **kNN вручную** — отсортировать по расстоянию/близости и взять top-k (никакой магии, чистый перебор).

### 📊 Что построить
- Таблица top-5 соседей по евклиду **и** по косинусу — сравните, где списки расходятся.
- 2D-проекция (любые два признака) с подсветкой цели и её соседей.
- Столбик цен: цель против найденных альтернатив.

### 🎤 Как защищать
Покажите 3 альтернативы и на сколько % они дешевле. Объясните, почему евклид и косинус выбрали разное — и какой смысл «похожести» ближе к запросу друга.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T4: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) FEATS = ["price_kgs","storage_gb","ram_gb","battery_pct"]  # выбор за вами
#   2) sub = df.dropna(subset=FEATS).copy()
#   3) z-нормировка: Z = (sub[FEATS] - sub[FEATS].mean()) / sub[FEATS].std()
#   4) выберите цель (например, iPhone 13 Pro) как вектор в Z
#   5) евклид: np.linalg.norm(Z.values - target, axis=1)
#      косинус: (Z @ target) / (норма строк * норма target)  # np.dot, np.linalg.norm
#   6) argsort -> top-5 соседей по каждой метрике, сравните

---
# 🧩 T5. Охота на перекупов

> На доске орудуют скупки и перекупы: рассрочка, «куплю дорого», десятки одинаковых объявлений с одного аккаунта. Вычислите их — и проверьте, портят ли они картину рынка (например, задирают ли среднюю цену).

**🙂 Наивный взгляд.** Объявления как объявления, продавцы как продавцы. Один рынок, всё честно.

**🔬 А что видит исследователь.** Часть «продавцов» — это бизнес: сотни объявлений с одного `user_id`, спам-заголовки с эмодзи, фейковые цены вроде `1234567890`. «Магазин» — это не готовая метка в данных, а **признак, который нужно сконструировать** из улик. А потом — сравнить, чем цены перекупов отличаются от частников.

### Ваша задача
- Найдите аномально активных продавцов — кандидатов в магазины/перекупы.
- Соберите из улик простой «индекс подозрительности» и разметьте продавцов.
- Сравните цены перекупов и частников на одну и ту же популярную модель.

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Распределение числа объявлений на продавца (log-шкала) + перцентиль.** Тут тонкость: у большинства продавцов ровно 1 объявление, поэтому Q1=Q3=1 и правило 1.5·IQR **вырождается** (порог «>1» бессмыслен). Это сам по себе урок — на счётчиках с тяжёлым хвостом берут высокий перцентиль (топ-5–10%) или явный разрыв на гистограмме.
- **Вектор-улика из z-оценок.** Сложите стандартизованные признаки (`n_ads_in_data`, `title_has_emoji`, экстремальность цены…) в один индекс подозрительности — это буквально длина/сумма компонент вектора.
- **Перцентили цен.** Фейковые цены живут в верхнем «хвосте»; порогом по перцентилю их удобно ловить.
- **Сравнение групп боксплотом.** Перекуп vs частник по цене одной модели — есть ли систематическая разница.

### 📊 Что построить
- Боксплот/гистограмма `n_ads_in_data` — где обрывается «частник» и начинается «магазин».
- Разметка продавцов по индексу подозрительности (сколько нашли).
- Цена одной модели: перекупы против частников.

### 🎤 Как защищать
Оцените, сколько на доске перекупов и как они искажают цену. Покажите 2–3 самых наглых объявления, которые ваш индекс поймал.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T5: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) активность продавцов: df["n_ads_in_data"].describe()
#      гистограмма в log-шкале: plt.hist(...); plt.yscale("log")
#   2) порог "магазина": высокий перцентиль (напр. .90) или явный разрыв
#      (правило 1.5*IQR тут вырождается — посмотрите на квартили, поймёте почему)
#   3) индекс подозрительности: сумма z-оценок улик
#      (n_ads_in_data, title_has_emoji, насколько цена экстремальна ...)
#   4) разметьте is_shop и сравните цену одной модели: перекуп vs частник
#   5) выведите top пойманных объявлений (title, price_kgs, n_ads_in_data)

---
## 🎤 Как оформить защиту (памятка)

Выступление — 3 минуты. Не пересказ кода, а **история одного решения**:

1. **Вопрос кейса** одной фразой + ваш **наивный ответ** (тот, что записали в начале).
2. **Что показали данные.** 3–5 графиков. Каждый график отвечает на вопрос «и что?».
3. **Поворот.** Где интуиция обманула? Один слайд «казалось → оказалось».
4. **Вердикт.** Конкретное решение и цифры. Не «айфоны дороже», а «берём модель X: медиана Y, батарея Z, риск минимален».
5. **Честность.** Что осталось за кадром, где данные слабые, чему бы вы не доверяли.

**Чек-лист хорошего графика:** есть заголовок; подписаны оси; видно единицы (сомы, ГБ, %); нет «каши»; **виден вывод**, а не просто «данные».

> Помните правило дома: комментарии в коде — по-английски, тексты и подписи — по-русски.

Удачи. Данные грязные — тем ценнее чистый вывод. 🚀